In [ ]:
import os
from dotenv import load_dotenv
import duckdb

load_dotenv()

DUCKDB_FILE = os.getenv('DUCKDB_FILE')

def duckdb_run_query(query: str):
    con = duckdb.connect(DUCKDB_FILE) # pyright: ignore[reportArgumentType]
    result = con.execute(query).fetchdf()
    con.close()
    return result

# VPIP (Voluntarily Put Money In Pot)

## Calculation:
Hands with Voluntary Bets (1) /  ( Total Hands (2) - Walks (3) )

1. These are calling the blinds, raising or betting, calling a bet.
2. Total number of hands in a given time range.
3. Player is the big blind but table folds.



Hands found and tested:
* 59107073baace709e358e55cd6ffb110 (OK)
* 43f14afb185500b82c3f77f79d84c07c (OK - had to remove fold actions from the query)
* f35e1bd5be1de039766c4b90b0d06e98 (OK)
* b47ce09df22665558ff892e56b6c7ae0 (OK position 3 is in the result set but I am worried that is a fluke. They call the blins as a first action.)

In [ ]:
df_handfinder = duckdb_run_query(f"""
                                 SELECT hand_id, COUNT(*) AS Actions
                                 FROM slv_actions
                                 WHERE street = 'preflop'
                                 GROUP BY Hand_id
                                 HAVING COUNT(*) > 5
                                 LIMIT 500
                                 """)

display(df_handfinder)

In [ ]:
hand_id = 'b47ce09df22665558ff892e56b6c7ae0'

In [ ]:
df_VPIP = duckdb_run_query(f"""
    SELECT slv_actions.player , action_seq, position,
        CASE WHEN slv_players.position = 1 THEN 'Small Blind'
             WHEN slv_players.position = 2 THEN 'Big Blind'
            ELSE NULL END AS Blind_Position,
        MAX(CASE WHEN action = 'raise' THEN 1 ELSE 0 END)
            OVER(PARTITION BY slv_actions.hand_id
                ORDER BY slv_actions.action_seq
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS raised_before_action,
    FROM slv_actions
        LEFT JOIN slv_players ON slv_actions.player = slv_players.player_name
            AND slv_actions.hand_id = slv_players.hand_id
    WHERE street = 'preflop' 
        AND slv_actions.hand_id = '{hand_id}'
        AND action != 'fold'
    ORDER BY action_seq
    LIMIT 500
""")

display(df_VPIP)

In [ ]:
display(duckdb_run_query(f"""SELECT slv_actions.action, slv_players.seat, slv_players.position,
                         FROM slv_actions 
                         LEFT JOIN slv_players ON slv_actions.player = slv_players.player_name
                            AND slv_actions.hand_id = slv_players.hand_id 
                         WHERE slv_actions.Hand_id ='{hand_id}' 
                            AND street = 'preflop'
                        ORDER BY slv_actions.action_seq
                            """))